<p style="text-align:center">
        <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
</p>


### Analyse search terms on the e-commerce web server


##### In this assignment you will download the search term data set for the e-commerce web server and run analytic queries on it.


## Install spark

In [1]:
!pip install pyspark
!pip install findspark
!pip install wget

### Setup and Spark Session
- initialize Spark environment and create the session.

In [2]:
import findspark
findspark.init()

In [3]:
from pyspark.sql import SparkSession

# Creating a spark session
spark = SparkSession \
    .builder \
    .appName("Analyse search terms on the e-commerce web server").getOrCreate()

26/03/09 11:25:05 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Download The search term dataset from the below url
- https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

In [4]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

--2026-03-09 11:25:13--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 233457 (228K) [text/csv]
Saving to: ‘searchterms.csv’

searchterms.csv     100%[===================>] 227.99K  --.-KB/s    in 0.01s   

2026-03-09 11:25:13 (20.7 MB/s) - ‘searchterms.csv’ saved [233457/233457]



## Load the Data & Basic Exploration
load the CSV file into a Spark DataFrame, count the rows, and check the schema to see the data types.

In [5]:
# Load the csv into a dataframe
df = spark.read.csv("searchterms.csv", header=True, inferSchema=True)

In [6]:
# Print the number of rows and columns
print(f"Total number of rows: {df.count()}")
print(f"Total number of columns: {len(df.columns)}")

Total number of rows: 10000
Total number of columns: 4


In [7]:
# Find out the datatype of the column 'searchterm'
df.printSchema()

root
 |-- day: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- searchterm: string (nullable = true)



In [8]:
# Print the top 5 rows
df.show(5)

+---+-----+----+--------------+
|day|month|year|    searchterm|
+---+-----+----+--------------+
| 12|   11|2021| mobile 6 inch|
| 12|   11|2021| mobile latest|
| 12|   11|2021|   tablet wifi|
| 12|   11|2021|laptop 14 inch|
| 12|   11|2021|     mobile 5g|
+---+-----+----+--------------+
only showing top 5 rows



### Analyze Search Terms
The notebook asks you to find specific insights from the search logs.

In [9]:
# How many times was the term `gaming laptop` searched?
gaming_laptop_count = df.filter(df["searchterm"] == "gaming laptop").count()
print(f"The term 'gaming laptop' was searched {gaming_laptop_count} times.")

The term 'gaming laptop' was searched 499 times.


In [10]:
# Print the top 5 most frequently used search terms?
print("Top 5 search terms:")
df.groupBy("searchterm") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(5)

Top 5 search terms:


+-------------+-----+
|   searchterm|count|
+-------------+-----+
|mobile 6 inch| 2312|
|    mobile 5g| 2301|
|mobile latest| 1327|
|       laptop|  935|
|  tablet wifi|  896|
+-------------+-----+
only showing top 5 rows



### Download and Load the Pre-trained Model
download the forecasting model provided by IBM, extract it, and load it into memory.

In [11]:
# Download the pre-trained sales forecasting model
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz

--2026-03-09 11:26:07--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1490 (1.5K) [application/x-tar]
Saving to: ‘model.tar.gz’

model.tar.gz        100%[===================>]   1.46K  --.-KB/s    in 0s      

2026-03-09 11:26:08 (14.1 MB/s) - ‘model.tar.gz’ saved [1490/1490]



In [12]:
# Extract the tar.gz file
!tar -xzvf model.tar.gz

sales_prediction.model/
sales_prediction.model/metadata/
sales_prediction.model/metadata/part-00000
sales_prediction.model/metadata/.part-00000.crc
sales_prediction.model/metadata/_SUCCESS
sales_prediction.model/metadata/._SUCCESS.crc
sales_prediction.model/data/
sales_prediction.model/data/part-00000-1db9fe2f-4d93-4b1f-966b-3b09e72d664e-c000.snappy.parquet
sales_prediction.model/data/_SUCCESS
sales_prediction.model/data/.part-00000-1db9fe2f-4d93-4b1f-966b-3b09e72d664e-c000.snappy.parquet.crc
sales_prediction.model/data/._SUCCESS.crc


In [13]:
# Load the sales forecast model.
from pyspark.ml.regression import LinearRegressionModel

model = LinearRegressionModel.load('sales_prediction.model')
print("Model loaded successfully!")

Model loaded successfully!


### Predict Sales for 2023
Finally, use the loaded model to predict the sales for the upcoming year.

In [ ]:
# Using the sales forecast model, predict the sales for the year of 2023.
from pyspark.ml.feature import VectorAssembler

# Define the prediction function
def predict(year):
    # Prepare the feature vector (models expect input as 'features')
    assembler = VectorAssembler(inputCols=["year"], outputCol="features")
    
    # Create a temporary dataframe for the year we want to predict
    data = [[year, 0]]
    columns = ["year", "sales"]
    temp_df = spark.createDataFrame(data, columns)
    
    # Transform the dataframe using the assembler
    features_df = assembler.transform(temp_df).select('features', 'year')
    
    # Run the prediction
    predictions = model.transform(features_df)
    
    print(f"Predicted Sales for the year {year}:")
    predictions.select('prediction').show()

In [15]:
# Predict sales for 2023
predict(2023)

Predicted Sales for the year 2023:
+------------------+
|        prediction|
+------------------+
|175.16564294006457|
+------------------+



26/03/09 11:26:53 WARN netlib.BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
26/03/09 11:26:53 WARN netlib.BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
